# Render English audio in Don's cloned voice (F5-TTS on Colab GPU)

**Before running anything:**

1. `Runtime → Change runtime type → T4 GPU → Save`
2. Then run each cell in order.

Expected total time: ~45–75 min on a T4. Don't close the browser tab during the render — Colab will disconnect.


## 1. Verify GPU

Output should mention `Tesla T4` (or similar). If it errors, the runtime isn't GPU — fix the Runtime type first.

In [ ]:
!nvidia-smi

## 2. Install prereqs (~2 min)

Node for the render orchestrator, f5-tts for the cloned-voice synthesis. ffmpeg is already on Colab's PATH.

In [ ]:
# Install a modern Node (Ubuntu's default nodejs package is too old
# for fs.rmSync and other APIs the render script uses).
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs -qq > /dev/null
!node --version
!pip install -q f5-tts

## 3. Clone the branch

Shallow clone of the multilingual-audio branch, then `cd` into it so subsequent cells run against the right tree.

In [ ]:
!git clone -b claude/improve-blog-audio-experience-wdFhv --depth 1 https://github.com/Donwonmagic/potentially-profitable.git
%cd potentially-profitable

## 4. Render all 8 English MP3s (~45–75 min on T4)

Each chunk should complete in 2–4 seconds on GPU. Progress lines scroll past as chunks finish — format is `chunk_id audio_seconds wall_clock_seconds`. If you see wall-clock > 20 s per chunk something's wrong; paste the output and we'll debug.

**Do not close this browser tab while this cell runs.**

In [ ]:
!node scripts/render-post-audio.mjs --all --engine f5 --languages en

## 5. Zip the audio and download

Creates a ~45 MB zip of the 16 newly-generated files (8 MP3s + 8 JSON manifests) and triggers a browser download. Drop it into `~/Downloads` on your Mac when prompted.

In [ ]:
import glob, os, zipfile
out = '/content/f5-english-audio.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for pattern in ['blog/*/audio.mp3', 'blog/*/audio.json',
                    'blog/drafts/*/audio.mp3', 'blog/drafts/*/audio.json']:
        for f in sorted(glob.glob(pattern)):
            z.write(f)
            print('+', f)
print(f'\nZipped {os.path.getsize(out) // 1024 // 1024} MB \u2192 {out}')

from google.colab import files
files.download(out)

## 6. Back on your Mac — commit and push

Run these in Terminal:

```bash
cd ~/potentially-profitable
unzip -o ~/Downloads/f5-english-audio.zip
git add blog/*/audio.mp3 blog/*/audio.json blog/drafts/*/audio.mp3 blog/drafts/*/audio.json
git commit -m "Re-render English audio in Don's cloned voice via F5-TTS"
git push
```

Done. The runtime player picks up the new MP3s automatically — no HTML edits needed.